학습목표

- HTTP의 무상태 특징을 이해하고 인증의 필요성을 안다.
- 쿠키와 세션의 동작 원리 및 데이터 저장 방식 차이를 안다.
- Django 기본 User 모델의 한계를 알고 커스텀 모델을 만든다
- AbstractUser를 상속받고 AUTH_USER_MODEL을 설정한다.
- AuthenticationForm과 login 함수로 로그인 기능을 구현한다.
- 로그인 시 세션이 생성되고 쿠키로 ID가 전달됨을 안다.
- request.user 객체로 로그인 상태에 따라 다른 화면을 표시한다.

Cookie & Session

HTTP (HyperText Transfer Protocol)
- HTML 문서와 같은 리소스들ㅇ르 가져올 수 있도록 해주는 규약 (웹에서의 모든 데이터 교환의 기초)

```
HTTP는 웹 브라우저와 서버가 서로 대화하기 위해 사용하는 공통 언어 또는 약속입니다.
브라우저가 '이 페이지 보여줘'라고 요청(Request)을 보내면, 서버는 그에 맞는 HTML, 이미지 등을 응답(Response)으로 보내주는 방식으로 동작합니다.
우리가 웹사이트를 보고, 이미지를 다운로드하고, 데이터를 주고받는 모든 과정이 바로 이 HTTP라는 규칙 위에서 이루어집니다.
```

HTTP 특징

1. 비 연결 지향(connectionless)
   - 서버는 요청에 대한 응답을 보낸 후 연결을 끊음
   - 클라이언트는 서버와 서로 연결되어 있는 상태가 아님

```
HTTP가 비 연결 지향으로 설계된 이유는?
- 서버가 문서를 다 읽을 때까지 모든 사용자와의 연결을 계속 유지해야 한다면, 수많은 연결이 서버의 메모리와 자원을 차지하게 됩니다.
- 이러한 자원 낭비를 막기 위해 HTTP는 다음과 같은 비연결(Connectionless) 방식을 채택했습니다.
```

2. 무상태(stateless)
   - 연결을 끊는 순간 클라이언트와 서버 간의 통신이 끝나며 상태 정보가 유지되지 않음
   - 무상태의 의미
     - 장바구니에 담은 상품을 유지할 수 없음
     - 로그인 상태를 유지할 수 없음

```
HTTP가 무상태(Stateless)로 설계된 이유는?
- 서버가 모든 클라이언트 상태를 기억한다고 하면 저장하고, 관리해야 하므로 매우 복잡해집니다.
- 더 큰 문제는 여러 대의 서버를 운영할 때, 클라이언트 상태를 공유하기 위해서 서로 다른 서버가 연결되어야 하는 문제로 인해서 확장성이 저하됩니다.
- 결국 무상태는 서버의 부담을 없애고 어떤 서버든 자유롭게 요청을 처리할 수 있게 만들어, 대규모 웹 서비스를 구축하는 데 핵심적인 역할을 합니다.
```



쿠키(Cookie)
- 서버가 사용자의 웹 브라우저에 전송하는 작은 데이터 조각

```
쿠키는 웹 사이트가 사용자으 부라우저에 남기는 작은 '데이터 조각'입니다.
이를 통해 서버는 '나는 이전에 이 사이트에 방문 했었고, 로그인도 했어'와 같이 사용자를 기억하고 식별할 수 있습니다.
덕분에 매번 아이디와 비밀번호를 다시 입력할 필요 없이 자동 로그인이 유지되거나, 장바구니에 담은 상품이 저장되는 것입니다.
```

쿠키 특징
- 서버가 사용자의 웹 브라우저에 전송하는 작은 데이터 조각
- 사용장 인증, 추적, 상태 유지 등에 사용되는 데이터 저장 방식
- key-value 형식의 데이터

쿠키 사용 예시
- 로그인 유지(세션 관리)
- 장바구니
- 언어, 테마 등 사용자 설정 기억

쿠키 동작 예시

1. 브라우저가 웹 서버에 웹 페이지를 요청
2. 웹 서버는 요청된 페이지와 함께 쿠키를 포함한 응답을 브라우저에게 전송
3. 브라우저는 받은 쿠키를 저장소에 저장하고, 쿠키의 속성(만료 시간, 도메인, 주소 등)도 함께 저장됨
4. 이후 브라우저가 같은 웹 서버에 웹 페이지를 요청할 때, 저장된 쿠키 중 해당 요청에 적용 가능한 쿠키를 포함하여 함께 전송
5. 웹 서버는 받은 쿠키 정보를 확인하고, 필요에 따라 사용자 식별, 세션 관리 들을 수행
6. 웹 서버는 요청에 대한 응답을 보내며, 필요한 경우 새로운 쿠키를 설정하거나 기존 쿠키를 수정할 수 있음

쿠키의 작동 원리와 활용

1. 쿠키 저장 방식
   - 브라우저(클라이언트)는 쿠키를 KEY-VALUE의 데이터 형식으로 저장
   - 쿠키에는 이름, 값 외에도 만료시간, 도메인, 경로 등의 추가 속성이 포함 됨
2. 쿠키 전송 과정
   - 서버는 HTTP 응답 헤더의 Set-Cookie 필드를 통해 클라이언트에게 쿠키를 전송
   - 브라우저는 받은 쿠키를 저장해 두었다가, 동일한 서버에 재요청 시 HTTP 요청 Header의 Cookie 필드에 저장된 쿠키를 함께 전송
3. 쿠키의 주요 용도
   - 두 요청이 동일한 브라우저에서 들어왔는지 아닌지를 판단할 때 주로 사용됨
   - 이를 이용해 사용자의 로그인 상태를 유지할 수 있음
   - 상태가 없는(stateless) HTTP 프로토콜에서 상태 정보를 기억시켜 주는 역할

> 서버에게 "나 로그인 된(인증된) 사용자야!"라는 인증 정보가 담긴 쿠키를 매 요청마다 계속 보내는 것

쿠키 사용 목적

1. 세션 관리(Session management)
   - 로그인, 아이디 자동완성, 공지 하루 안 보기, 팝업 체크, 장바구니 등의 정보 관리
2. 개인화 (Personalization)
   - 사용자 선호 설정(언어 설정, 테마 등) 저장
3. 추적, 수집(Tracking)
   - 사용자 행동을 기록 및 분석

```
- 쿠키는 탈취도리 수 있으니, 비밀번호 등 민감한 정보는 절대 직접 저장하면 안 됩니다.
- 쿠키는 모든 요청에 포함되어 전송되므로, 크기를 최소화해야 사이트 성능에 유리합니다.
```



세션(session)

- 서버 측에서 생성되어 클라이언트와 서버 간의 상태를 유지, 상태 정보를 저장하는 데이터 저장 방식

```
세션은 로그인 정보와 같은 중요 데이터를 클라이언트가 아닌 서버 쪽에 저장하고 유지하는 기술입니다.
서버는 각 사용자를 구분하기 위해 고유한 '세션 ID'를 발급하고, 이 ID만 쿠키에 담아 클라이언트에 보내 사용자를 식별합니다.
실제 데이터는 서버에만 보관되므로 쿠키만 사용하는 방식보다 훨씬 보안에 유리하며, 사용자의 로그인 상태를 안전하게 유지하는 데 줄 사용됩니다.
```

세션 동작 원리

1. 클라이언트가 로그인 요청 후 인증에 성공하면 서버가 session 데이터를 생성 후 저장
2. 생성된 session 데이터에 인증 할 수 있는 session id를 발급
3. 발급한 session id를 클라이언트에게 응답(데이터는 서버에 저장, 열쇠만 주는 것)
4. 클라이언트는 응답 받은 session id를 쿠키에 저장
5. 클라이언트가 다시 동일한 서버에 접속하면 요청과 함께 쿠키(session id가 저장된)f를 서버에 전달
6. 쿠키는 요청 때마다 서버에 함께 전송 되므로 서버에서 session id를 확인해 로그인 되어있다는 것을 계속해서 확인하도록 함
7. 사용자의 요청을 처리하고 응답

세션 특징
- 서버 측에서 생성되어 클라이언트와 서버 간의 상태를 유지
  - 서버의 메모리나 데이터베이스에 저장되므로, 서버 리소스를 사용(효율적 관리 필요)
- 상태 정보를 저장하는 데이터 저장 방식
- 쿠키에 세션 데이터를 저장하여 매 요청시마다 세션 데이터를 함께 보냄
- 세션은 영구적으로 유지되지 않음

```
- 중요한 데이터를 저장함으로 보안을 신겨써야 함
- 공격자가 세션 ID를 탈취하면, 해당 사용자인 겻처럼 위장하여 서버에 접근이 가능하므로 유의
```

세션 정리

1. 서버 측에서는 세션 데이터를 생성 후 저장하고, 이 데이터에 접근할 수 있는 세션 ID를 생성
2. 이 ID를 클라이언트 측으로 전달하고, 클라이언트는 쿠키에 이 ID를 저장
3. 이후 클라이언트가 같은 서버에 재요청 시마다 저장해 두었던 쿠키도 요청과 함께 전송
   - 로그인 상태 유지를 위해 로그인 되어있다는 사실을 입증하는 데이터를 매 요청마다 계속해서 보내는 것


쿠키와 세션의 목적
- 클라이언트와 서버 간의 상태 정보를 유지하고 사용자를 식별하기 위해 사용

Django Authentication System

인증의 필요성
- 클라이언트와 서버 간의 상태 정보를 유지하기 위해서 쿠키와 세션을 사용
- 클라이언트와 서버는 각기 다른 사용자를 식별해야 하는 상태
- 그래서 사용자를 식별하기 위해서 필요한 과정이 바로 '인증(Autentication)'
- 다양한 인증이 존재
  - 아이디와 비밀번호
  - 소셜 로그인
  - 생체인증

> Django에서는 사용자 인증과 관련된 가장 중요하고 기본적인 뼈대를 제공 (Django Autentication System)

Django Autentication System

- Django에서 사용자 인증과 관련된 기능을 모아 놓은 시스템
- 인증에 중요한 기본적인 기능을 제공
  - User Model : 사용자 인증 후 연결될 User Model 관리
  - Session 관리 : 로그인 상태를 유지하고 서버에 저장하는 방식을 관리
  - 기본 인증(id/password): 로그인/로그아웃 등 다양한 기능을 제공

기본 User Model의 한계
- 인증 후 사용된느 User Model은 별도의 User클래스 정의 없이 내장된 auth 앱에 작성된 User 클래스를 사용했음
- 하지만 Django의 기본 User모델은 username, password 등 제공되는 필드가 매우 제한적
- 추가적인 사용자 정보(예: 생년월일, 주소, 나이 등)가 필요하다면 이를 위해 기본 User Model을 변경하기 어려움
  - 별도의 설정 없이 사용할 수 있어 간편하지만, 개발자가 직접 수정하기 어려움

내장된 auth 앱
- 기본적으로 username, password, email 등의 필드를 가진 User 모델을 제공
- 단순히 로그인 여부만 확인하는 것을 넘어, 사용자별 또는 그룹별로 특정 행동에 대한 권한 부여 가능

User Model 대체의 필요성
- 프로젝트의 특정 요구사항에 맞춰 사용자 모델을 확장할 수 있음
- 예를 들어
  - 이메일을 username으로 사용하거나, 다른 추가 필드를 포함시킬 수 있음
  - 기본 User 모델의 first_name, last_name처럼 우리 프로젝트에 필요 없는 필드를 제거하여 데이터베이스 모델을 더 간결하게 관리할 수 있음

> Django에서 제공하는 기본 유저 모델이 아닌 우리가 직접 커스텀한 유저 모델을 사용해보자

Custom User model

User model 대체하기

사전 준비
- 두 번째 app accounts 생성 및 등록
- auth와 관련된 경로나 키워드들을 django 내부적으로 accounts라는 이름으로 사용하고 있기 때문에 되도록 'accounts'로 지정하는 것을 권장

Custom User Model로 대체하기
- AbstractUser 클래스를 상속받는 커스텀 User 클래스 작성

> 기존 User 클래스도 AbstractUser를 상속받기 때문에 커스텀 User 클래스도 기존 User 클래스와 완전히 같은 모습을 가지게 됨  

- django 프로젝트에서 사용하는 기본 User 모델을 우리가 작성한 User 모델로 사용할 수 있도록 AUTH_USER_MODEL 값을 변경
  - accounts 앱에 작성한 User 모델을 기본 모델로 설정

- admin site에 대체한 User 모델 등록
  - 기본 User 모델이 아니기 때문에 등록하지 않으면 admin페이지에 출력되지 않기 때문

AUTH_USEr_MODEL
- Django 프로젝트의 User를 나타내는 데 사용하는 모델을 지정하는 속성

```
* 주의 *
Django는 프로젝트 중간에 AUTH_USER_MODEL을 변경하는 것을 강력하게 권장하지 않음
```

> 이미 프로젝트가 진행되고 있을 경우 데이터베이스 초기화 후 진행

사용하는 User 테이블의 변화
- accounts에 만든 커스텀 유저 모델이 데이터베이스에 반영 됨

프로젝트를 시작하며 반드시 User 모델을 대체해야 함
- Django는 새 프로젝트를 시작하는 경우 비록 기본 User 모델이 충분하더라도 커스텀 User 모델을 설정하는 것을 강력하게 권장하고 있음
- 커스텀 User 모델은 기본 User 모델과 동일하게 작동(1) 하면서도 나중에 맞춤 설정(2)할 수 있음

> 단, User 모델 대체 작업은 프로젝트의 모든 migrations혹은 첫 migrate를 실행하기 전에 이 작업을 마쳐야 함

```
- 지금 당장 필요 없어도 만드세요. 나중에 닉네임 등 필드를 추가하기가 매우 쉬워집니다.
- 모델 생성 후, settings.py의 AUTH_USER_MODEL을 설정해야만 Django가 이 모델을 사용합니다.

Login
- 클라이언트와 서버 간의 상태 정보를 유지하기 위해서 쿠키와 세션을 사용
- 클라이언트와 서버는 각기 다른 사용자를 식별해야 하는 상태
- 서버에 '나'임을 인증하는 과정이 바로 'Login'

> 결국 Login은 인증(id/password)을 완료하고, Session을 만들고 클라이언트와 연결하는 것

로그인 기능 구현
- 로그인 경로 url 및 링크 생성

- .../accounts/login/rul 로 요청이 들어왔을 때 실행할 login 함수 작성
- 로그인 인증에 사용할 데이터를 입력 받는 built-in form(AuthenticationForm)사용

- 로그인 정보를 서버에 안전하게 전송하게 위해 'POST방식'을 사용
- CSRF 공격을 방지하기 위해 csrf_token 작성
- 서버로부터 전달받은 AuthenticationForm을 화면에 출력

- 사용자가 입력한 로그인 정보를 입력받은 로직을 'POST' 조건문에 작성
- 입력 받은 정보를 기반으로 로그인하여 세션을 만드는 'login' 함수를 활용


AuthenticationForm()
- 로그인 인증에 사용할 데이터를 입력 받는 built-in form
- User 모델과 직접 연결된 'ModelForm'이 아닌, 일반 'Form'을 상속 받음
  - 일반 'Form'이기 때문에 사용자를 생성하거나 수정하는 용도가 아닌 '인증'하는 역할만 수행